# 01 — Reproduce Submitted Figures

```text
Reviewer concern addressed: None directly; establishes that the submitted baseline
    (Figure 1, Supp. Fig. S1, Supp. Fig. S2) can be regenerated before any revision
    analysis changes the story. Directly follows from 00_environment_check.ipynb,
    which found the active kernel imports HiMaLAYAS 0.0.16a0 (editable, dirty,
    unreleased) while the README pins 0.0.15.
Input files: fig_1.ipynb, supp_fig_1.ipynb, supp_fig_2.ipynb (read-only; never
    modified) + the five data files hashed in 00_environment_check_manifest.json
HiMaLAYAS version: TWO sources are used and clearly labeled -- the verified git tag
    v0.0.15 from the upstream package repo ("pinned_v0_0_15", treated as the
    submitted baseline) and the active kernel's editable install ("dev_0_0_16a0",
    explicitly NOT the baseline, comparison only). See Section 2.
Random seed: none required; root notebooks use deterministic Ward/Euclidean
    clustering with optimal_ordering=True (confirmed in 00_environment_check).
Primary parameters: none introduced here; each root notebook's own clustering/
    enrichment parameters are executed unmodified.
Outputs written:
    revision/outputs/manifests/01_reproduce_submitted_figures_manifest.json
    revision/outputs/figures/01_reproduce_submitted_figures/<mode>/<notebook>/*.png
    revision/outputs/tables/01_reproduce_submitted_figures/<mode>/<notebook>/*.csv
    revision/outputs/executed_notebooks/<notebook>.<mode>.executed.ipynb
Interpretation: see the Readiness Summary in the final section.
```

## Policy: what counts as "the submitted baseline" in this notebook

`00_environment_check.ipynb` found that the active kernel imports HiMaLAYAS
`0.0.16a0` via an editable install of a sibling development repo, while this
repo's README pins `himalayas==0.0.15` and the vendored `himalayas_src/` copy
also *declares* `0.0.15`.

**This notebook does not treat that active `0.0.16a0` environment as a valid
submitted-version reproduction.** Instead:

- **Baseline = the exact source at git tag `v0.0.15`** in the upstream
  `himalayas` package repository, verified read-only (`git show`/`git
  archive`) and extracted into a scratch directory for import. This is
  treated as ground truth because it is a tag, not a version string.
- **`himalayas_src/` (vendored copy) is *not* assumed to equal that tag.** It
  is checked byte-for-byte against the tag first (Section 2). If it diverges,
  the tag is used instead of the vendored copy for the "pinned" run.
- **The active `0.0.16a0` environment is run *only* as a clearly labeled
  development-version comparison**, never presented as the baseline.

## 1. Locate repo root and import shared revision utilities

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    markers = ("himalayas_src", "data", ".git", "revision")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f"Could not locate himalayas-publication repo root above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "revision" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Kernel CWD:   {Path.cwd()}")
print(f"Repo root:    {REPO_ROOT}")

Kernel CWD:   /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/notebooks
Repo root:    /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication


In [2]:
import json
import traceback
from datetime import datetime, timezone

import pandas as pd

import revision_utils as ru
from revision_utils import revision_layout

layout = revision_layout(REPO_ROOT)
for key in ("manifests_dir", "figures_dir", "tables_dir", "executed_notebooks_dir", "scratch_dir"):
    layout[key].mkdir(parents=True, exist_ok=True)

NB01_FIGURES_DIR = layout["figures_dir"] / "01_reproduce_submitted_figures"
NB01_TABLES_DIR = layout["tables_dir"] / "01_reproduce_submitted_figures"
NB01_SCRATCH_DIR = layout["scratch_dir"] / "01_reproduce_submitted_figures"
for d in (NB01_FIGURES_DIR, NB01_TABLES_DIR, NB01_SCRATCH_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Figures ->", NB01_FIGURES_DIR)
print("Tables  ->", NB01_TABLES_DIR)
print("Scratch ->", NB01_SCRATCH_DIR)

Figures -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/01_reproduce_submitted_figures
Tables  -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/01_reproduce_submitted_figures
Scratch -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/01_reproduce_submitted_figures


## 2. Recall notebook 00's findings, then re-verify the environment now

00's manifest is loaded for provenance, but the HiMaLAYAS diagnostics are re-computed fresh (the environment could have changed since 00 ran).

In [3]:
nb00_manifest_path = layout["manifests_dir"] / "00_environment_check_manifest.json"
nb00_manifest = ru.read_manifest(nb00_manifest_path)

print("00_environment_check flags recorded previously:")
for f in nb00_manifest["flags"]:
    print(f"  - {f}")

himalayas_diag = ru.himalayas_diagnostics(layout["repo_root"])
print()
print("Re-checked just now:")
print(json.dumps(himalayas_diag, indent=2))

if himalayas_diag["imported_version"] != nb00_manifest["himalayas"]["imported_version"]:
    print(
        "\nNOTE: active HiMaLAYAS version has changed since 00 ran "
        f"({nb00_manifest['himalayas']['imported_version']!r} -> "
        f"{himalayas_diag['imported_version']!r})."
    )

00_environment_check flags recorded previously:
  - ACTIVE HiMaLAYAS VERSION MISMATCH: kernel imports '0.0.16a0' but README pins '0.0.15'.
  - EDITABLE HiMaLAYAS SOURCE IS DIRTY: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas has 5 uncommitted path(s): ['M src/himalayas/__init__.py', '?? PREVIEW_NOTES.md', '?? _archive/', '?? preview-release-notes.sh', '?? test.py'].
  - EDITABLE HiMaLAYAS SOURCE IS AHEAD OF ITS LAST TAG: git describe = 'v0.0.15-4-g0c51115'.
  - VENDORED COPY VERSION DIFFERS FROM IMPORTED VERSION: himalayas_src/ declares '0.0.15', kernel imports '0.0.16a0'.
  - PUBLICATION REPO HAS UNCOMMITTED/UNTRACKED PATHS: 6 path(s) -- expected during active revision work (e.g. this new revision/ tree itself); noted for provenance only.



Re-checked just now:
{
  "imported_version": "0.0.16a0",
  "imported_file": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas/src/himalayas/__init__.py",
  "is_editable_install": true,
  "editable_project_location": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas",
  "editable_git": {
    "is_git_repo": true,
    "branch": "v0.0.16",
    "commit": "0c511157663c7f392d9338e101830c462ce59b9c",
    "describe": "v0.0.15-4-g0c51115",
    "is_dirty": true,
    "dirty_files": [
      "M src/himalayas/__init__.py",
      "?? PREVIEW_NOTES.md",
      "?? _archive/",
      "?? preview-release-notes.sh",
      "?? test.py"
    ]
  },
  "vendored_copy_path": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/himalayas_src/himalayas/__init__.py",
  "vendored_copy_version": "0.0.15",
  "readme_pinned_version": "0.0.15",
  "matches_readme_pinned_ver

## 3. Verify candidate "0.0.15" sources against the ground-truth git tag

Two candidates claim to be `0.0.15`: the vendored `himalayas_src/` copy in this repo, and (if reachable) an upstream sibling repo's `v0.0.15` git tag. A version string is not proof of identity -- compare file contents directly.

In [4]:
sibling_repo = himalayas_diag["editable_project_location"]
PINNED_TAG = "v0.0.15"

tag_available = False
if sibling_repo is not None:
    sibling_repo = Path(sibling_repo)
    tag_available = ru.repo_has_tag(sibling_repo, PINNED_TAG)

print(f"Sibling dev repo (editable install location): {sibling_repo}")
print(f"Sibling repo reachable and has tag {PINNED_TAG!r}: {tag_available}")

vendored_diff = None
if tag_available:
    vendored_diff = ru.diff_vendored_against_tag(
        sibling_repo, PINNED_TAG, layout["repo_root"] / "himalayas_src"
    )
    print(
        f"\nhimalayas_src/ vs tag {PINNED_TAG}: "
        f"{vendored_diff['files_compared']} files compared, "
        f"{len(vendored_diff['mismatched_files'])} mismatched"
    )
    if vendored_diff["mismatched_files"]:
        print("Mismatched files:")
        for p in vendored_diff["mismatched_files"]:
            print(f"  - {p}")
else:
    print(
        "\nCannot verify himalayas_src/ against the tag on this machine "
        "(no reachable sibling repo with that tag)."
    )

Sibling dev repo (editable install location): /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas
Sibling repo reachable and has tag 'v0.0.15': True



himalayas_src/ vs tag v0.0.15: 29 files compared, 7 mismatched
Mismatched files:
  - core/annotations.py
  - plot/condensed_dendrogram.py
  - plot/plotter.py
  - plot/renderers/axes.py
  - plot/renderers/label_legend.py
  - plot/renderers/matrix.py
  - plot/style.py


In [5]:
source_verification_flags = []

if not tag_available:
    source_verification_flags.append(
        f"PINNED SOURCE UNVERIFIABLE ON THIS MACHINE: sibling repo with tag "
        f"{PINNED_TAG!r} not reachable at {sibling_repo!r}. This reproduction "
        "method is not portable to Binder/CI as written -- see Readiness Summary."
    )
elif not vendored_diff["matches_tag_exactly"]:
    source_verification_flags.append(
        f"VENDORED himalayas_src/ DOES NOT MATCH TAG {PINNED_TAG}: "
        f"{len(vendored_diff['mismatched_files'])}/{vendored_diff['files_compared']} files differ "
        "(non-trivial Plotter/renderer changes, not just typing modernization). "
        "himalayas_src/ declares 0.0.15 but is a later, unreleased snapshot. "
        f"DECISION: use the verified tag {PINNED_TAG} (extracted fresh below), "
        "not himalayas_src/, as the pinned baseline import source for this notebook."
    )
else:
    source_verification_flags.append(
        f"VENDORED himalayas_src/ MATCHES TAG {PINNED_TAG} exactly ({vendored_diff['files_compared']} files)."
    )

for f in source_verification_flags:
    print(f"- {f}\n")

- VENDORED himalayas_src/ DOES NOT MATCH TAG v0.0.15: 7/29 files differ (non-trivial Plotter/renderer changes, not just typing modernization). himalayas_src/ declares 0.0.15 but is a later, unreleased snapshot. DECISION: use the verified tag v0.0.15 (extracted fresh below), not himalayas_src/, as the pinned baseline import source for this notebook.



## 4. Extract a verified-pristine v0.0.15 source tree

Extracted via `git archive` (read-only against the sibling repo) into `revision/scratch/` -- a regenerable build artifact, not evidence in itself. The manifest, tables, and figures produced below are the evidence.

In [6]:
pinned_src_dir = None
extraction_error = None

if tag_available:
    pinned_extract_root = NB01_SCRATCH_DIR / "himalayas_pinned_v0_0_15"
    try:
        pinned_src_dir = ru.extract_tagged_source(
            sibling_repo, PINNED_TAG, pinned_extract_root, subpath="src"
        )
        # Sanity self-check: the freshly extracted tree must match the tag by construction.
        self_check = ru.diff_vendored_against_tag(
            sibling_repo, PINNED_TAG, pinned_src_dir, vendored_subpath="himalayas"
        )
        assert self_check[
            "matches_tag_exactly"
        ], "extracted tree unexpectedly differs from its own tag"
        print(f"Extracted verified {PINNED_TAG} source to: {pinned_src_dir}")
        print(f"Self-check against tag: {self_check['files_compared']} files, all match.")
    except Exception as exc:  # noqa: BLE001
        extraction_error = f"{type(exc).__name__}: {exc}"
        print(f"EXTRACTION FAILED: {extraction_error}")
else:
    print("Skipping extraction: no reachable sibling repo with the pinned tag.")

Extracted verified v0.0.15 source to: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/01_reproduce_submitted_figures/himalayas_pinned_v0_0_15/src
Self-check against tag: 29 files, all match.


## 5. Execution harness: run each root notebook under each HiMaLAYAS source

Each root notebook is loaded read-only, copied in memory, and executed with (a) `sys.path` pinned to a specific HiMaLAYAS source and (b) its `PNG_DIR` redirected out of the tracked `png/` directory at repo root and into `revision/outputs/figures/01_reproduce_submitted_figures/`. The original `fig_1.ipynb` / `supp_fig_1.ipynb` / `supp_fig_2.ipynb` files are never written to.

In [7]:
ROOT_NOTEBOOKS = ["fig_1.ipynb", "supp_fig_1.ipynb", "supp_fig_2.ipynb"]

MODES = []
if pinned_src_dir is not None:
    MODES.append(
        {
            "label": "pinned_v0_0_15",
            "sys_path_prepend": pinned_src_dir,
            "expected_version": "0.0.15",
            "description": f"Verified git tag {PINNED_TAG} -- treated as the submitted baseline.",
            "is_baseline": True,
        }
    )
MODES.append(
    {
        "label": "dev_0_0_16a0",
        "sys_path_prepend": None,
        "expected_version": himalayas_diag["imported_version"],
        "description": (
            "Active kernel editable install -- DEVELOPMENT VERSION reproduction only, "
            "NOT the submitted baseline."
        ),
        "is_baseline": False,
    }
)

for m in MODES:
    print(f"{m['label']:16s} baseline={m['is_baseline']!s:5s} {m['description']}")

pinned_v0_0_15   baseline=True  Verified git tag v0.0.15 -- treated as the submitted baseline.
dev_0_0_16a0     baseline=False Active kernel editable install -- DEVELOPMENT VERSION reproduction only, NOT the submitted baseline.


In [8]:
run_records = []

for nb_name in ROOT_NOTEBOOKS:
    nb_stem = Path(nb_name).stem
    source_nb = ru.load_root_notebook(layout["repo_root"] / nb_name)

    for mode in MODES:
        label = mode["label"]
        figure_dir = NB01_FIGURES_DIR / label / nb_stem
        export_dir = NB01_TABLES_DIR / label / nb_stem

        run_nb = ru.build_run_copy(
            source_nb,
            figure_dir=figure_dir,
            export_dir=export_dir,
            sys_path_prepend=mode["sys_path_prepend"],
        )

        t0 = datetime.now(timezone.utc)
        executed_nb, error = ru.execute_notebook(run_nb, cwd=layout["repo_root"])
        duration_s = (datetime.now(timezone.utc) - t0).total_seconds()

        cell_error = ru.has_cell_error(executed_nb)
        text = ru.stream_text(executed_nb)

        import re as _re

        version_match = _re.search(r"HiMaLAYAS version:\s*(\S+)", text)
        printed_version = version_match.group(1) if version_match else None

        success = (error is None) and (not cell_error)
        version_ok = printed_version == mode["expected_version"]

        executed_path = layout["executed_notebooks_dir"] / f"{nb_stem}.{label}.executed.ipynb"
        nbf_module = __import__("nbformat")
        nbf_module.write(executed_nb, executed_path)

        figures = (
            sorted(str(p.relative_to(layout["repo_root"])) for p in figure_dir.glob("*.png"))
            if figure_dir.exists()
            else []
        )
        figure_hashes = {Path(p).name: ru.sha256_file(layout["repo_root"] / p) for p in figures}

        record = {
            "notebook": nb_name,
            "mode": label,
            "is_baseline": mode["is_baseline"],
            "description": mode["description"],
            "success": success,
            "expected_himalayas_version": mode["expected_version"],
            "printed_himalayas_version": printed_version,
            "version_matches_expected": version_ok,
            "duration_seconds": round(duration_s, 2),
            "error": error,
            "had_cell_error": cell_error,
            "executed_notebook_path": str(executed_path.relative_to(layout["repo_root"])),
            "figure_dir": str(figure_dir.relative_to(layout["repo_root"])),
            "figures": figures,
            "figure_sha256": figure_hashes,
            "export_dir": str(export_dir.relative_to(layout["repo_root"])),
        }
        run_records.append(record)
        status = "OK" if success and version_ok else "FAIL" if not success else "VERSION-MISMATCH"
        print(
            f"[{status:16s}] {nb_name:18s} mode={label:16s} {duration_s:5.1f}s  figures={len(figures)}"
        )
        if not success:
            print(f"    error: {error}")
            if cell_error:
                for i, c in enumerate(executed_nb.cells):
                    for out in c.get("outputs", []):
                        if out.get("output_type") == "error":
                            print(f"    cell {i}: {out.get('ename')}: {out.get('evalue')}")

[OK              ] fig_1.ipynb        mode=pinned_v0_0_15     4.7s  figures=2


[OK              ] fig_1.ipynb        mode=dev_0_0_16a0       4.8s  figures=2


[OK              ] supp_fig_1.ipynb   mode=pinned_v0_0_15    13.2s  figures=5


[OK              ] supp_fig_1.ipynb   mode=dev_0_0_16a0      13.1s  figures=5


[OK              ] supp_fig_2.ipynb   mode=pinned_v0_0_15     7.5s  figures=3


[OK              ] supp_fig_2.ipynb   mode=dev_0_0_16a0       7.3s  figures=3


## 6. Cross-mode comparison: does the development version change the reported results?

Pixel-identical PNGs are not a realistic bar across matplotlib/font versions -- figure dimensions and a mean-pixel-difference are reported for context, but the decisive check is the underlying analysis output (`results`, `results_sig`, `cluster_labels`), exported to CSV by every run above.

In [9]:
def _record_for(nb_name, label):
    return next(r for r in run_records if r["notebook"] == nb_name and r["mode"] == label)


comparison_rows = []
baseline_label = (
    next(m["label"] for m in MODES if m["is_baseline"])
    if any(m["is_baseline"] for m in MODES)
    else None
)
other_labels = [m["label"] for m in MODES if not m["is_baseline"]]

if baseline_label is not None:
    for nb_name in ROOT_NOTEBOOKS:
        base_rec = _record_for(nb_name, baseline_label)
        for other_label in other_labels:
            other_rec = _record_for(nb_name, other_label)
            row = {
                "notebook": nb_name,
                "baseline_mode": baseline_label,
                "compared_mode": other_label,
            }

            for target in ("results", "results_sig", "cluster_labels"):
                base_csv = layout["repo_root"] / base_rec["export_dir"] / f"{target}.csv"
                other_csv = layout["repo_root"] / other_rec["export_dir"] / f"{target}.csv"
                row[f"{target}_rows_baseline"] = ru.csv_row_count(base_csv)
                row[f"{target}_rows_compared"] = ru.csv_row_count(other_csv)
                row[f"{target}_content_identical"] = ru.csv_content_matches(base_csv, other_csv)

            base_figs = {Path(p).name for p in base_rec["figures"]}
            other_figs = {Path(p).name for p in other_rec["figures"]}
            shared = sorted(base_figs & other_figs)
            dim_matches, pixel_diffs = [], []
            for fname in shared:
                base_path = layout["repo_root"] / base_rec["figure_dir"] / fname
                other_path = layout["repo_root"] / other_rec["figure_dir"] / fname
                dim_a = ru.image_dimensions(base_path)
                dim_b = ru.image_dimensions(other_path)
                dim_matches.append(dim_a == dim_b)
                pixel_diffs.append(ru.pixel_mean_abs_diff(base_path, other_path))
            row["figures_compared"] = len(shared)
            row["figure_dims_all_match"] = all(dim_matches) if dim_matches else None
            row["figure_pixel_mean_abs_diff"] = (
                sum(d for d in pixel_diffs if d is not None) / len(pixel_diffs)
                if pixel_diffs and any(d is not None for d in pixel_diffs)
                else None
            )
            comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,notebook,baseline_mode,compared_mode,results_rows_baseline,results_rows_compared,results_content_identical,results_sig_rows_baseline,results_sig_rows_compared,results_sig_content_identical,cluster_labels_rows_baseline,cluster_labels_rows_compared,cluster_labels_content_identical,figures_compared,figure_dims_all_match,figure_pixel_mean_abs_diff
0,fig_1.ipynb,pinned_v0_0_15,dev_0_0_16a0,709,709,True,331,331,True,7,7,True,2,True,0.0
1,supp_fig_1.ipynb,pinned_v0_0_15,dev_0_0_16a0,984,984,True,464,464,True,12,12,True,5,True,0.0
2,supp_fig_2.ipynb,pinned_v0_0_15,dev_0_0_16a0,91,91,True,18,18,True,8,8,True,3,True,0.0


## 7. Informational check against the pre-existing `png/` directory

`png/` at the repo root is **untracked** (never committed; `git log --all -- png/` is empty) and of unknown provenance -- it may have been rendered under an earlier environment or an earlier version of a notebook. It is *not* treated as verified submitted output, only compared for informal continuity. Any dimension mismatch is cross-checked against git history: if the existing PNG predates the notebook's last edit commit, that alone explains a rendering difference without implying a reproduction problem.

In [10]:
import subprocess


def _last_commit_timestamp(repo_root, rel_path):
    try:
        out = subprocess.run(
            ["git", "-C", str(repo_root), "log", "-1", "--format=%aI", "--", rel_path],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
        return out or None
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


existing_png_rows = []
if baseline_label is not None:
    for nb_name in ROOT_NOTEBOOKS:
        nb_stem = Path(nb_name).stem
        base_rec = _record_for(nb_name, baseline_label)
        existing_dir = layout["repo_root"] / "png" / nb_stem
        notebook_last_commit = _last_commit_timestamp(layout["repo_root"], nb_name)
        for fname in [Path(p).name for p in base_rec["figures"]]:
            existing_path = existing_dir / fname
            new_path = layout["repo_root"] / base_rec["figure_dir"] / fname
            existing_mtime_dt = (
                datetime.fromtimestamp(existing_path.stat().st_mtime, tz=timezone.utc)
                if existing_path.exists()
                else None
            )
            existing_mtime = (
                existing_mtime_dt.isoformat() if existing_mtime_dt is not None else None
            )
            notebook_last_commit_dt = (
                datetime.fromisoformat(notebook_last_commit) if notebook_last_commit else None
            )
            predates_edit = (
                existing_mtime_dt is not None
                and notebook_last_commit_dt is not None
                # Compare as timezone-aware instants, not ISO strings: string comparison across
                # different UTC offsets (e.g. "+00:00" vs "-04:00") does not preserve ordering.
                and existing_mtime_dt < notebook_last_commit_dt
            )
            existing_png_rows.append(
                {
                    "notebook": nb_name,
                    "figure": fname,
                    "existing_png_present": existing_path.exists(),
                    "existing_png_dims": ru.image_dimensions(existing_path),
                    "regenerated_dims": ru.image_dimensions(new_path),
                    "dims_match": ru.image_dimensions(existing_path)
                    == ru.image_dimensions(new_path),
                    "sha256_identical": (
                        ru.sha256_file(existing_path) == ru.sha256_file(new_path)
                        if existing_path.exists()
                        else None
                    ),
                    "existing_png_mtime_utc": existing_mtime,
                    "notebook_last_commit_utc": notebook_last_commit,
                    "existing_png_predates_notebook_edit": predates_edit,
                }
            )

existing_png_df = pd.DataFrame(existing_png_rows)
existing_png_df

,notebook,figure,existing_png_present,existing_png_dims,regenerated_dims,dims_match,sha256_identical,existing_png_mtime_utc,notebook_last_commit_utc,existing_png_predates_notebook_edit
0,fig_1.ipynb,yeast_gi_similarity_condensed_dendrogram.png,True,"(2229, 1769)","(2307, 1803)",False,False,2026-03-30T19:44:52.984201+00:00,2026-03-30T16:59:38-04:00,True
1,fig_1.ipynb,yeast_gi_similarity_matrix.png,True,"(2934, 3631)","(2948, 3658)",False,False,2026-03-30T19:44:54.086690+00:00,2026-03-30T16:59:38-04:00,True
2,supp_fig_1.ipynb,cluster_7_zoom_matrix.png,True,"(2018, 4099)","(2018, 4099)",True,True,2026-05-13T17:58:18.490491+00:00,2026-07-17T15:09:07-04:00,True
3,supp_fig_1.ipynb,cluster_8_zoom_matrix.png,True,"(1101, 4222)","(1101, 4222)",True,True,2026-05-13T17:58:20.255833+00:00,2026-07-17T15:09:07-04:00,True
4,supp_fig_1.ipynb,cluster_9_zoom_matrix.png,True,"(2576, 4402)","(2576, 4402)",True,True,2026-05-13T17:58:22.355847+00:00,2026-07-17T15:09:07-04:00,True
5,supp_fig_1.ipynb,yeast_gi_score_condensed_dendrogram.png,True,"(1767, 4003)","(1767, 4003)",True,True,2026-05-13T17:58:16.123985+00:00,2026-07-17T15:09:07-04:00,True
6,supp_fig_1.ipynb,yeast_gi_score_matrix.png,True,"(3631, 3639)","(3631, 3639)",True,True,2026-05-13T17:58:15.707553+00:00,2026-07-17T15:09:07-04:00,True
7,supp_fig_2.ipynb,cluster_feature_usage_grid.png,True,"(1797, 5923)","(1797, 5923)",True,True,2026-05-13T17:58:24.375206+00:00,2026-03-30T12:22:47-04:00,False
8,supp_fig_2.ipynb,world_recipe_condensed_dendrogram.png,True,"(2577, 1969)","(2577, 1969)",True,True,2026-05-13T17:58:23.893005+00:00,2026-03-30T12:22:47-04:00,False
9,supp_fig_2.ipynb,world_recipe_matrix.png,True,"(3423, 4028)","(3553, 4028)",False,False,2026-05-13T17:58:23.716160+00:00,2026-03-30T12:22:47-04:00,False


In [11]:
mismatched = existing_png_df[~existing_png_df["dims_match"]]
if mismatched.empty:
    print("All existing png/ figures match the regenerated baseline dimensions exactly.")
else:
    print(f"{len(mismatched)} figure(s) differ in dimensions from the existing png/ directory:\n")
    for _, row in mismatched.iterrows():
        print(f"  {row['notebook']} / {row['figure']}")
        print(
            f"    existing dims={row['existing_png_dims']}  regenerated dims={row['regenerated_dims']}"
        )
        if row["existing_png_predates_notebook_edit"]:
            print(
                f"    LIKELY EXPLANATION: existing PNG rendered {row['existing_png_mtime_utc']}, "
                f"*before* the notebook's last edit commit ({row['notebook_last_commit_utc']}) -- "
                "the existing PNG is stale relative to the current notebook source, not a "
                "reproduction failure."
            )
        else:
            print(
                "    Existing PNG postdates (or has no determinable relation to) the notebook's "
                "last edit -- dimension difference not explained by staleness; likely font/"
                "matplotlib-version rendering variation (see Section 6 pixel-diff notes)."
            )
        print()

3 figure(s) differ in dimensions from the existing png/ directory:

  fig_1.ipynb / yeast_gi_similarity_condensed_dendrogram.png
    existing dims=(2229, 1769)  regenerated dims=(2307, 1803)
    LIKELY EXPLANATION: existing PNG rendered 2026-03-30T19:44:52.984201+00:00, *before* the notebook's last edit commit (2026-03-30T16:59:38-04:00) -- the existing PNG is stale relative to the current notebook source, not a reproduction failure.

  fig_1.ipynb / yeast_gi_similarity_matrix.png
    existing dims=(2934, 3631)  regenerated dims=(2948, 3658)
    LIKELY EXPLANATION: existing PNG rendered 2026-03-30T19:44:54.086690+00:00, *before* the notebook's last edit commit (2026-03-30T16:59:38-04:00) -- the existing PNG is stale relative to the current notebook source, not a reproduction failure.

  supp_fig_2.ipynb / world_recipe_matrix.png
    existing dims=(3423, 4028)  regenerated dims=(3553, 4028)
    Existing PNG postdates (or has no determinable relation to) the notebook's last edit -- dimen

## 8. Assemble and write manifest

In [12]:
manifest = {
    "notebook": "01_reproduce_submitted_figures.ipynb",
    "generated_at_utc": ru.utc_timestamp(),
    "repo_root": str(layout["repo_root"]),
    "baseline_policy": {
        "baseline_definition": (
            f"HiMaLAYAS source at git tag {PINNED_TAG} in the upstream package "
            "repository, verified byte-for-byte via `git show`/`git archive` "
            "(read-only) before use. NOT the active kernel's editable install, "
            "and NOT assumed to equal the vendored himalayas_src/ copy without checking."
        ),
        "active_kernel_version": himalayas_diag["imported_version"],
        "active_kernel_is_baseline": False,
    },
    "nb00_flags_recalled": nb00_manifest["flags"],
    "himalayas_diagnostics_rechecked": himalayas_diag,
    "source_verification": {
        "sibling_repo": str(sibling_repo) if sibling_repo else None,
        "pinned_tag": PINNED_TAG,
        "tag_available": tag_available,
        "vendored_vs_tag_diff": vendored_diff,
        "flags": source_verification_flags,
        "pinned_src_extraction_error": extraction_error,
        "pinned_src_dir": str(pinned_src_dir) if pinned_src_dir else None,
    },
    "modes": [{k: (str(v) if isinstance(v, Path) else v) for k, v in m.items()} for m in MODES],
    "run_records": run_records,
    "cross_mode_comparison": comparison_rows,
    "existing_png_comparison": existing_png_rows,
}

manifest_path = layout["manifests_dir"] / "01_reproduce_submitted_figures_manifest.json"
ru.write_manifest(manifest, manifest_path)
print(f"Manifest written to: {manifest_path}")

reloaded = json.loads(manifest_path.read_text())
assert reloaded == json.loads(json.dumps(manifest, default=str)), "manifest round-trip mismatch"
print("Manifest JSON round-trip verified.")

Manifest written to: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/01_reproduce_submitted_figures_manifest.json
Manifest JSON round-trip verified.


## 9. Readiness summary

In [13]:
print("=" * 72)
print("READINESS SUMMARY -- 01_reproduce_submitted_figures")
print("=" * 72)

baseline_records = [r for r in run_records if r["is_baseline"]]
dev_records = [r for r in run_records if not r["is_baseline"]]

baseline_ok = bool(baseline_records) and all(
    r["success"] and r["version_matches_expected"] for r in baseline_records
)
any_baseline_attempted = bool(baseline_records)

if not any_baseline_attempted:
    status = "BLOCKED"
elif baseline_ok:
    status = "REPRODUCED"
elif any(r["success"] for r in baseline_records):
    status = "PARTIALLY REPRODUCED"
else:
    status = "BLOCKED"

print(f"STATUS: {status}\n")

print(f"Baseline source: verified git tag {PINNED_TAG} (tag_available={tag_available})")
if not tag_available:
    print(
        "  -> No reachable sibling repo with the pinned tag on this machine. "
        "This is a portability limitation: the method used here to obtain a "
        "verified 0.0.15 source depends on a local sibling git checkout, which "
        "will NOT exist on Binder/CI/another reviewer's machine. Not resolved "
        "in this notebook -- see recommendation below."
    )

print()
print("Per-notebook baseline (pinned_v0_0_15) results:")
for r in baseline_records:
    print(
        f"  {r['notebook']:18s} success={r['success']!s:5s} "
        f"version_ok={r['version_matches_expected']!s:5s} figures={len(r['figures'])}"
    )

print()
print("Per-notebook development-version (dev_0_0_16a0) results:")
for r in dev_records:
    print(
        f"  {r['notebook']:18s} success={r['success']!s:5s} "
        f"version_ok={r['version_matches_expected']!s:5s} figures={len(r['figures'])}"
    )

print()
if not comparison_df.empty:
    identical_cols = [c for c in comparison_df.columns if c.endswith("_content_identical")]
    all_identical = comparison_df[identical_cols].all(axis=None) if identical_cols else None
    print(
        f"Pinned (0.0.15) vs dev (0.0.16a0) analysis-level agreement, all notebooks identical: {all_identical}"
    )
    if all_identical is False:
        mismatched_notebooks = comparison_df.loc[
            ~comparison_df[identical_cols].all(axis=1), "notebook"
        ].tolist()
        print(f"  Notebooks with a data-level difference between versions: {mismatched_notebooks}")
        print("  -> The 0.0.16a0 development version changes reported results for at least")
        print("     one submitted figure. Treat dev-version output as informative only; do")
        print("     not substitute it for the baseline anywhere in the revision package.")
    elif all_identical is True:
        print("  -> 0.0.16a0 reproduces identical results/cluster tables to the verified")
        print("     0.0.15 baseline for all three submitted analyses (figures may still")
        print("     differ cosmetically; see dims/pixel-diff columns above).")
else:
    print("No cross-mode comparison available (baseline could not run).")

print()
print("Vendored himalayas_src/ status:")
for f in source_verification_flags:
    print(f"  - {f}")

print()
print("Outstanding action items (not resolved in this notebook, by design):")
print("  1. himalayas_src/ misrepresents its version (__version__='0.0.15') while diverging")
print("     from the actual v0.0.15 tag in 7/29 source files. Either refresh it to match")
print("     the tag exactly, or stop treating its version string as authoritative.")
print("  2. The pinned-baseline extraction method used here (git archive from a local")
print("     sibling repo) is not portable to Binder/CI. Before relying on this baseline")
print("     for reviewer-facing reproducibility claims, either vendor a verified-identical")
print("     0.0.15 copy into this repo, or confirm `pip install himalayas==0.0.15` resolves")
print("     to the same tagged source (not attempted here -- would require network access).")
print()
print("Recommendation for notebook 10 onward:")
print("  Continue importing HiMaLAYAS via the same verified-tag extraction approach")
print("  (revision_utils.extract_tagged_source / diff_vendored_against_tag) rather than")
print(
    "  relying on himalayas_src/ or the active editable install, until action item 1 is resolved."
)

READINESS SUMMARY -- 01_reproduce_submitted_figures
STATUS: REPRODUCED

Baseline source: verified git tag v0.0.15 (tag_available=True)

Per-notebook baseline (pinned_v0_0_15) results:
  fig_1.ipynb        success=True  version_ok=True  figures=2
  supp_fig_1.ipynb   success=True  version_ok=True  figures=5
  supp_fig_2.ipynb   success=True  version_ok=True  figures=3

Per-notebook development-version (dev_0_0_16a0) results:
  fig_1.ipynb        success=True  version_ok=True  figures=2
  supp_fig_1.ipynb   success=True  version_ok=True  figures=5
  supp_fig_2.ipynb   success=True  version_ok=True  figures=3

Pinned (0.0.15) vs dev (0.0.16a0) analysis-level agreement, all notebooks identical: True

Vendored himalayas_src/ status:
  - VENDORED himalayas_src/ DOES NOT MATCH TAG v0.0.15: 7/29 files differ (non-trivial Plotter/renderer changes, not just typing modernization). himalayas_src/ declares 0.0.15 but is a later, unreleased snapshot. DECISION: use the verified tag v0.0.15 (extracted